# AVO intercept-gradient analysis

This notebook calculates the AVO intercept and gradient for the corrected `DataAVO2.csv` dataset.

Only these two interfaces are evaluated:
- shale to brine
- shale to gas

The intercept-gradient crossplot also includes arrows from each shale-to-brine point to the corresponding shale-to-gas point so the fluid-substitution shift is easy to see.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from matplotlib.patches import Rectangle

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.float_format', lambda x: f'{x:0.4f}')


In [ ]:
data_path = '../Datasets/DataAVO2.csv'
df = (
    pd.read_csv(data_path)
    .rename(columns={'Shale_VP': 'Shale_Vp', 'Shale_Vs.1': 'Shale_p'})
)

display(df[['Index', 'Shale_Vp', 'Shale_Vs', 'Shale_p', 'Brine_Vp', 'Brine_Vs', 'Brine_p', 'Gas_Vp', 'Gas_Vs', 'Gas_p']].head())
print("Using shale density directly from the corrected CSV column 'Shale_Vs.1' renamed to 'Shale_p'.")


In [ ]:
def shuey_intercept_gradient(vp1, vs1, rho1, vp2, vs2, rho2):
    """Return Shuey two-term intercept (A) and gradient (B)."""
    vp_avg = 0.5 * (vp1 + vp2)
    vs_avg = 0.5 * (vs1 + vs2)
    rho_avg = 0.5 * (rho1 + rho2)

    d_vp = vp2 - vp1
    d_vs = vs2 - vs1
    d_rho = rho2 - rho1

    intercept = 0.5 * (d_vp / vp_avg + d_rho / rho_avg)
    gradient = 0.5 * (d_vp / vp_avg) - 2.0 * (vs_avg / vp_avg) ** 2 * (2.0 * d_vs / vs_avg + d_rho / rho_avg)
    return intercept, gradient

def classify_avo(intercept, gradient, near_zero=0.02):
    if abs(intercept) <= near_zero and gradient < 0:
        return 'Class II'
    if intercept > 0 and gradient < 0:
        return 'Class I'
    if intercept < 0 and gradient < 0:
        return 'Class III'
    if intercept < 0 and gradient > 0:
        return 'Class IV'
    return 'Unclassified'

def build_interface_table(frame, interface_name, vp_col, vs_col, rho_col, angles=np.arange(0, 41, 5)):
    records = []
    for row in frame.itertuples(index=False):
        intercept, gradient = shuey_intercept_gradient(
            row.Shale_Vp, row.Shale_Vs, row.Shale_p,
            getattr(row, vp_col), getattr(row, vs_col), getattr(row, rho_col)
        )
        reflectivity = intercept + gradient * np.sin(np.radians(angles)) ** 2
        records.append({
            'Index': row.Index,
            'Interface': interface_name,
            'Intercept': intercept,
            'Gradient': gradient,
            'AVO_Class': classify_avo(intercept, gradient),
            'R(0deg)': reflectivity[0],
            'R(20deg)': reflectivity[np.where(angles == 20)[0][0]],
            'R(40deg)': reflectivity[-1],
        })
    return pd.DataFrame(records)

brine_results = build_interface_table(df, 'Shale to Brine', 'Brine_Vp', 'Brine_Vs', 'Brine_p')
gas_results = build_interface_table(df, 'Shale to Gas', 'Gas_Vp', 'Gas_Vs', 'Gas_p')

avo_results = pd.concat([brine_results, gas_results], ignore_index=True)
shift_results = brine_results.merge(
    gas_results,
    on='Index',
    suffixes=('_Brine', '_Gas')
)

display(avo_results.head(10))
display(shift_results[['Index', 'Intercept_Brine', 'Gradient_Brine', 'Intercept_Gas', 'Gradient_Gas']].head(10))


In [ ]:
summary = (
    avo_results.groupby(['Interface', 'AVO_Class'])[['Intercept', 'Gradient']]
    .agg(['count', 'mean'])
    .round(4)
)
summary


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 8), constrained_layout=True)

angles = np.arange(0, 41, 5)
interface_styles = {
    'Shale to Brine': {'color': '#1f77b4', 'marker': 'o'},
    'Shale to Gas': {'color': '#d62728', 'marker': '^'},
}

# Left panel: AVO curves generated from intercept and gradient
for interface_name, subset in avo_results.groupby('Interface'):
    style = interface_styles[interface_name]
    for row in subset.itertuples(index=False):
        amplitude = row.Intercept + row.Gradient * np.sin(np.radians(angles)) ** 2
        axes[0].plot(angles, amplitude, color=style['color'], alpha=0.25)

axes[0].set_title('Shuey two-term reflectivity curves')
axes[0].set_xlabel('Incident angle (degrees)')
axes[0].set_ylabel('Reflection coefficient')
axes[0].axhline(0, color='black', linewidth=1)
axes[0].plot([], [], color='#1f77b4', linewidth=2, label='Shale to Brine')
axes[0].plot([], [], color='#d62728', linewidth=2, label='Shale to Gas')
axes[0].legend(loc='best')

# Right panel: intercept-gradient crossplot with class areas and fluid-shift arrows
brine_points = avo_results.loc[avo_results['Interface'] == 'Shale to Brine', ['Index', 'Intercept', 'Gradient']].copy()
gas_points = avo_results.loc[avo_results['Interface'] == 'Shale to Gas', ['Index', 'Intercept', 'Gradient']].copy()
shift_results = brine_points.merge(gas_points, on='Index', suffixes=('_Brine', '_Gas'))

x = avo_results['Intercept']
y = avo_results['Gradient']
x_limit = 1.10 * np.abs(x).max()
y_limit = 1.10 * np.abs(y).max()
xmin, xmax = -x_limit, x_limit
ymin, ymax = -y_limit, y_limit
class_ii_halfwidth = max(0.02, 0.12 * x_limit)

ax = axes[1]
ax.add_patch(Rectangle((class_ii_halfwidth, ymin), xmax - class_ii_halfwidth, -ymin, facecolor='#d6ebff', alpha=0.28, zorder=0))
ax.add_patch(Rectangle((xmin, ymin), -class_ii_halfwidth - xmin, -ymin, facecolor='#ffd9d2', alpha=0.28, zorder=0))
ax.add_patch(Rectangle((-class_ii_halfwidth, ymin), 2 * class_ii_halfwidth, -ymin, facecolor='#fff3bf', alpha=0.45, zorder=0))
ax.add_patch(Rectangle((xmin, 0), -xmin, ymax, facecolor='#d8f3dc', alpha=0.30, zorder=0))

for interface_name, subset in avo_results.groupby('Interface'):
    style = interface_styles[interface_name]
    ax.scatter(
        subset['Intercept'], subset['Gradient'],
        s=70, marker=style['marker'], c=style['color'],
        edgecolor='black', linewidth=0.6, alpha=0.98, label=interface_name, zorder=3
    )

for row in shift_results.itertuples(index=False):
    dx = row.Intercept_Gas - row.Intercept_Brine
    dy = row.Gradient_Gas - row.Gradient_Brine
    ax.plot(
        [row.Intercept_Brine, row.Intercept_Gas],
        [row.Gradient_Brine, row.Gradient_Gas],
        color='#202020',
        linewidth=0.9,
        alpha=0.28,
        zorder=2,
    )
    ax.annotate(
        '',
        xy=(row.Intercept_Gas, row.Gradient_Gas),
        xytext=(row.Intercept_Brine + 0.78 * dx, row.Gradient_Brine + 0.78 * dy),
        arrowprops=dict(
            arrowstyle='-|>',
            mutation_scale=9,
            lw=0.9,
            color='#202020',
            alpha=0.42,
            shrinkA=0,
            shrinkB=0,
        ),
        zorder=4,
    )

for row in brine_points.itertuples(index=False):
    ax.annotate(int(row.Index), (row.Intercept, row.Gradient), xytext=(4, 4), textcoords='offset points', fontsize=8, color='#1f1f1f', alpha=0.85)

ax.axhline(0, color='black', linewidth=1)
ax.axvline(0, color='black', linewidth=1)
ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)
ax.set_title('Intercept-gradient crossplot with fluid-shift arrows')
ax.set_xlabel('Intercept, A')
ax.set_ylabel('Gradient, B')
ax.set_aspect('equal', adjustable='box')
ax.legend(loc='upper right')
ax.text(0.03, 0.97, 'Arrow direction: brine to gas', transform=ax.transAxes, va='top', fontsize=10, color='#202020', bbox=dict(boxstyle='round,pad=0.25', facecolor='white', edgecolor='#202020', alpha=0.85))

ax.text(0.72 * xmax, 0.78 * ymin, 'Class I', fontsize=11, weight='bold', color='#22577a')
ax.text(0.0, 0.86 * ymin, 'Class II', fontsize=11, weight='bold', color='#8a5a00', ha='center')
ax.text(0.75 * xmin, 0.78 * ymin, 'Class III', fontsize=11, weight='bold', color='#9d0208')
ax.text(0.72 * xmin, 0.78 * ymax, 'Class IV', fontsize=11, weight='bold', color='#2d6a4f')

plt.show()


In [ ]:
avo_results.sort_values(['Interface', 'Index']).reset_index(drop=True)
